# Robot Path Planning Using Q-Learning
## Baseline Q-Learning vs. Guided / Reward-Shaped Q-Learning

**Research paper:** He Du et al., “A path planning approach for mobile robots using short and safe Q-learning,” *PLOS ONE*, 17(9), e0275100 (2022).  
**Open-access paper:** https://doi.org/10.1371/journal.pone.0275100  
**Code repository listed by the paper:** https://github.com/SCI-DHgithub/A-path-planning-approach-for-mobile-robots-using-short-and-safe-Q-learning/issues

This notebook is an **educational, reproducible project inspired by the paper**, not a line-by-line reproduction of the authors' full SSQL method. It compares tabular Q-learning with a guided variant that uses potential-field values to initialize Q-values and a distance-based reward. The environment is a small grid world with obstacles.


## 1. Objectives
- Train an agent to navigate from a start cell to a goal cell without hitting obstacles.
- Compare classical tabular Q-learning with a guided/reward-shaped variant.
- Report success rate, path length, and training reward.
- Visualize the environment and the learned route.

## 2. Install requirements
The notebook uses only common Python packages. If needed, run the next cell in Jupyter/Colab.


In [ ]:
%pip -q install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import time
from collections import deque

SEED = 7
random.seed(SEED)
np.random.seed(SEED)


## 3. Define the grid-world environment

The agent can move in four directions (up, down, left, right). Each step has a small cost, reaching the goal gives a positive reward, and hitting an obstacle or boundary gives a penalty.


In [ ]:
ROWS, COLS = 10, 10
START = (0, 0)
GOAL = (9, 9)

# 1 = obstacle, 0 = free cell
GRID = np.zeros((ROWS, COLS), dtype=int)
obstacles = [
    (1, 0), (1, 1), (1, 2), (1, 3),
    (3, 2), (3, 3), (3, 4), (3, 5),
    (5, 4), (5, 5), (5, 6),
    (7, 1), (7, 2), (7, 3), (7, 7), (8, 7)
]
for cell in obstacles:
    GRID[cell] = 1
GRID[START] = 0
GRID[GOAL] = 0

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = ["Up", "Down", "Left", "Right"]

def step(state, action_idx):
    dr, dc = ACTIONS[action_idx]
    nr, nc = state[0] + dr, state[1] + dc
    # Invalid move: remain in the same cell and receive a penalty.
    if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or GRID[nr, nc] == 1:
        return state, -5.0, False
    next_state = (nr, nc)
    if next_state == GOAL:
        return next_state, 100.0, True
    return next_state, -1.0, False

def plot_grid(path=None, title="Grid world"):
    arr = GRID.astype(float).copy()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(arr, cmap="Greys", vmin=0, vmax=1)
    ax.scatter(START[1], START[0], s=180, marker="o", label="Start")
    ax.scatter(GOAL[1], GOAL[0], s=220, marker="*", label="Goal")
    if path:
        ys = [p[1] for p in path]
        xs = [p[0] for p in path]
        ax.plot(ys, xs, linewidth=2.5, marker=".", label="Learned path")
    ax.set_xticks(range(COLS)); ax.set_yticks(range(ROWS))
    ax.set_xticklabels(range(COLS)); ax.set_yticklabels(range(ROWS))
    ax.grid(True, color="lightgray", linewidth=0.6)
    ax.set_title(title); ax.legend()
    plt.show()

plot_grid()


## 4. Optional potential-field initialization

The paper combines artificial potential field (APF) information with Q-learning to guide initial learning. This simplified version assigns a higher potential to cells closer to the goal and a repulsive penalty near obstacles. We use the potential to initialize the Q-table for the guided agent.

This is a compact demonstration of the idea, not the paper's exact APF equations.


In [ ]:
def potential(state):
    r, c = state
    # Attractive potential: lower distance to goal => higher desirability.
    goal_dist = abs(r - GOAL[0]) + abs(c - GOAL[1])
    attractive = -0.8 * goal_dist

    # Repulsive potential from nearby obstacles.
    repulsive = 0.0
    for rr, cc in obstacles:
        d = abs(r - rr) + abs(c - cc)
        if d == 0:
            repulsive -= 20
        elif d <= 2:
            repulsive -= 2.0 / d
    return attractive + repulsive

def make_initial_q(guided=False):
    q = np.zeros((ROWS, COLS, len(ACTIONS)), dtype=float)
    if guided:
        for r in range(ROWS):
            for c in range(COLS):
                if GRID[r, c] == 0:
                    s = (r, c)
                    for a, (dr, dc) in enumerate(ACTIONS):
                        ns = (r + dr, c + dc)
                        if 0 <= ns[0] < ROWS and 0 <= ns[1] < COLS and GRID[ns] == 0:
                            q[r, c, a] = 0.15 * potential(ns)
    return q

plot_grid()


## 5. Q-learning training function

Q-learning update:

\[
Q(s,a) \leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right]
\]

We use an epsilon-greedy policy: the agent explores randomly with probability epsilon and otherwise chooses the action with the highest Q-value. For the guided agent, the reward includes a small distance-progress term.


In [ ]:
def train_q_learning(guided=False, episodes=2500, max_steps=250,
                     alpha=0.15, gamma=0.95, epsilon=1.0,
                     epsilon_min=0.03, epsilon_decay=0.997):
    q = make_initial_q(guided=guided)
    episode_returns = []
    episode_lengths = []
    successes = []

    for ep in range(episodes):
        state = START
        total_reward = 0.0
        success = False

        for t in range(max_steps):
            if random.random() < epsilon:
                action = random.randrange(len(ACTIONS))
            else:
                action = int(np.argmax(q[state[0], state[1], :]))

            next_state, reward, done = step(state, action)

            if guided and not done:
                # Reward shaping: positive when Manhattan distance decreases.
                d_before = abs(state[0]-GOAL[0]) + abs(state[1]-GOAL[1])
                d_after = abs(next_state[0]-GOAL[0]) + abs(next_state[1]-GOAL[1])
                reward += 0.8 * (d_before - d_after)

            old = q[state[0], state[1], action]
            if done:
                target = reward
            else:
                target = reward + gamma * np.max(q[next_state[0], next_state[1], :])
            q[state[0], state[1], action] = old + alpha * (target - old)

            state = next_state
            total_reward += reward
            if done:
                success = True
                break

        episode_returns.append(total_reward)
        episode_lengths.append(t + 1)
        successes.append(int(success))
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return q, np.array(episode_returns), np.array(episode_lengths), np.array(successes)

start_time = time.time()
q_base, returns_base, lengths_base, success_base = train_q_learning(guided=False)
base_seconds = time.time() - start_time

start_time = time.time()
q_guided, returns_guided, lengths_guided, success_guided = train_q_learning(guided=True)
guided_seconds = time.time() - start_time

print(f"Baseline training time: {base_seconds:.2f} s")
print(f"Guided training time:  {guided_seconds:.2f} s")


## 6. Extract a route from the learned Q-table

The route is extracted greedily from the learned policy. A visited-state guard prevents infinite loops if training has not learned a complete route.


In [ ]:
def extract_path(q, max_steps=200):
    state = START
    path = [state]
    seen = {state}
    for _ in range(max_steps):
        action = int(np.argmax(q[state[0], state[1], :]))
        next_state, reward, done = step(state, action)
        if next_state == state or next_state in seen:
            return path, False
        path.append(next_state)
        if done:
            return path, True
        seen.add(next_state)
        state = next_state
    return path, False

path_base, ok_base = extract_path(q_base)
path_guided, ok_guided = extract_path(q_guided)

print("Baseline reached goal:", ok_base, "| path cells:", len(path_base))
print("Guided reached goal:  ", ok_guided, "| path cells:", len(path_guided))
plot_grid(path_base, "Classical Q-learning route")
plot_grid(path_guided, "Guided / reward-shaped Q-learning route")


## 7. Plot learning curves

Curves are noisy, so we plot a moving average. The exact results can vary slightly with random seeds and parameter settings.


In [ ]:
def moving_average(x, window=100):
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window)/window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(moving_average(returns_base), label="Classical Q-learning")
plt.plot(moving_average(returns_guided), label="Guided / shaped Q-learning")
plt.xlabel("Episode")
plt.ylabel("Moving-average return")
plt.title("Training reward comparison")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
window = 100
base_rate = moving_average(success_base.astype(float), window)
guided_rate = moving_average(success_guided.astype(float), window)
plt.plot(base_rate, label="Classical Q-learning success rate")
plt.plot(guided_rate, label="Guided / shaped Q-learning success rate")
plt.xlabel("Episode (moving window)")
plt.ylabel("Success fraction")
plt.title("Success rate during training")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True)
plt.show()


## 8. Quantitative evaluation

For a fairer comparison, evaluate each learned policy over multiple episodes. Since the environment is deterministic, the same greedy policy should generally produce the same route each time; the repeated evaluation is mainly useful if you later add stochastic transitions.


In [ ]:
def evaluate_policy(q, trials=100):
    successes = 0
    path_lengths = []
    for _ in range(trials):
        path, success = extract_path(q)
        successes += int(success)
        if success:
            path_lengths.append(len(path) - 1)
    return {
        "success_rate_percent": 100 * successes / trials,
        "mean_path_length_steps": float(np.mean(path_lengths)) if path_lengths else np.nan,
        "successful_trials": successes,
        "trials": trials
    }

results = {
    "Classical Q-learning": evaluate_policy(q_base),
    "Guided / shaped Q-learning": evaluate_policy(q_guided)
}
for name, result in results.items():
    print("\n", name)
    for k, v in result.items():
        print(f"  {k}: {v}")


## 9. Optional: change the experiment

Try changing one variable at a time and record the outcome:
- `episodes`: 500, 1000, 2500, 5000
- `alpha`: 0.1, 0.15, 0.3
- `gamma`: 0.8, 0.9, 0.95
- `epsilon_decay`: 0.99, 0.995, 0.997
- Add or remove obstacle cells

Run each configuration with several random seeds for more reliable comparisons. Report the actual outputs from your run; do not copy assumed results into the report.


## 10. Limitations and academic integrity

- This is a small grid-world simulation, not a physical robot experiment.
- The guided algorithm is a simplified implementation inspired by the paper's APF-initialization and dynamic-reward ideas; it does not reproduce every equation, safe-distance rule, map, or baseline in the published study.
- In your report, clearly distinguish the paper's published findings from your own experimental results.
- Cite the original paper and its repository. If you claim to reproduce the exact SSQL method, implement and verify the paper's full equations and experiment settings first.

### Reference
Du, H., Hao, B., Zhao, J., Zhang, J., Wang, Q., & Yuan, Q. (2022). *A path planning approach for mobile robots using short and safe Q-learning*. PLOS ONE, 17(9), e0275100. https://doi.org/10.1371/journal.pone.0275100
